In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.4 Dataset splits

You cannot measure learning on the data you trained on. Hold out a validation
slice and never train on it: the gap between train and val loss is what tells you
whether the model generalizes or just memorizes.

In [ ]:
from pocketlm import CharTokenizer
corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)

from pocketlm import split_data
train, val = split_data(data, val_frac=0.1)
print("train tokens:", len(train), " val tokens:", len(val))
print("contiguous, no overlap:", len(train) + len(val) == len(data))

Leakage, val data sneaking into training, makes a model look great and fail in
the wild. A clean contiguous split is the simplest defense.

In [ ]:
assert len(train) + len(val) == len(data)
assert abs(len(val) / len(data) - 0.1) < 0.01